# 🛡️ AI SHIELD — IndoBERT Fine-Tuning Pipeline
### *Smart Handling of Integrity in Ethical Live Dialogue*

**Program:** PIJAK in collaboration with IBM SkillsBuild — Dicoding  
**Tema:** AI for Productivity and Automation

---

## 📋 Alur Pengembangan AI (sesuai Flowchart AI/ML Engineer)

Notebook ini mengikuti alur resmi yang terdefinisi dalam `flowchart.md` dan `project-plan.md`:

| Langkah | Tahap | Keterangan |
|---------|-------|------------|
| 1️⃣ | **Pengumpulan Dataset** | Unduh dataset dari GitHub okkyibrohim |
| 2️⃣ | **EDA (Exploratory Data Analysis)** | Analisis distribusi label, panjang teks, bias |
| 3️⃣ | **Preprocessing Teks** | Lowercase, hapus URL/emoji, normalisasi slang |
| 4️⃣ | **Relabeling ke Biner** | Konversi multi-label → `PANTAS` / `TIDAK PANTAS` |
| 5️⃣ | **Split Dataset** | Train 70% · Validation 15% · Test 15% |
| 6️⃣ | **Fine-tuning IndoBERT** | PyTorch + HuggingFace Trainer |
| 7️⃣ | **Evaluasi Model** | Accuracy, Precision, Recall, F1, Confusion Matrix |
| 8️⃣ | **Kalibrasi Threshold** | Tentukan confidence threshold optimal (default 0.75) |
| 9️⃣ | **Inference Function** | Fungsi `predict(text)` → `{label, confidence}` |
| 🔟 | **Simpan & Export Model** | Simpan ke Google Drive untuk digunakan Backend |

**Target Performa:**
- ✅ Accuracy ≥ 85%
- ✅ F1-Score kelas `TIDAK PANTAS` ≥ 82%

> ⚠️ **Pastikan runtime Google Colab diatur ke GPU (T4/A100)** sebelum menjalankan notebook ini.
> Menu: `Runtime` → `Change runtime type` → `GPU`

---
## ⚙️ LANGKAH 0 — Setup Environment & Install Dependencies

In [ ]:
# ============================================================
# LANGKAH 0: Install semua library yang dibutuhkan
# Jalankan sekali, restart runtime jika diminta
# ============================================================

print("📦 Menginstall library yang dibutuhkan...")

!pip install -q transformers==4.40.0
!pip install -q datasets==2.19.0
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install -q scikit-learn==1.4.2
!pip install -q pandas numpy matplotlib seaborn
!pip install -q accelerate==0.29.3
!pip install -q evaluate==0.4.1
!pip install -q requests

print("✅ Semua library berhasil diinstall!")

In [ ]:
# ============================================================
# Import semua library yang diperlukan
# ============================================================

import os
import re
import json
import random
import warnings
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from io import StringIO
from collections import Counter

import torch
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    roc_curve,
    auc
)

import evaluate

warnings.filterwarnings('ignore')

# Atur seed agar hasil dapat direproduksi
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

# Cek GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Device yang digunakan: {device}")
if torch.cuda.is_available():
    print(f"🚀 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️  GPU tidak tersedia. Training akan SANGAT lambat di CPU.")
    print("   Silakan aktifkan GPU di: Runtime → Change runtime type → GPU")

print("\n✅ Import sukses dan environment siap!")

---
## 📥 LANGKAH 1 — Pengumpulan Dataset

Mengunduh dataset **Indonesian Multi-Label Hate Speech & Abusive Language** dari GitHub:
- Repository: [okkyibrohim/id-multi-label-hate-speech-and-abusive-language-detection](https://github.com/okkyibrohim/id-multi-label-hate-speech-and-abusive-language-detection)
- File utama: `re_data.csv` — data tweet berlabel hate speech & abusive
- Referensi: Ibrohim & Budi (2019) — *Multi-label Hate Speech Detection in Indonesian Twitter*

In [ ]:
# ============================================================
# LANGKAH 1: Download Dataset dari GitHub
# ============================================================

# URL raw file di GitHub
GITHUB_BASE = "https://raw.githubusercontent.com/okkyibrohim/id-multi-label-hate-speech-and-abusive-language-detection/master"

# File-file dataset yang akan diunduh
FILES = {
    "data.csv": f"{GITHUB_BASE}/re_data.csv",
    "alay_dict.csv": f"{GITHUB_BASE}/new_kamusalay.csv",
    "stopwords.txt": f"{GITHUB_BASE}/stopwordbahasa.csv"
}

os.makedirs("dataset", exist_ok=True)

for filename, url in FILES.items():
    filepath = f"dataset/{filename}"
    if not os.path.exists(filepath):
        print(f"⬇️  Mengunduh {filename}...")
        try:
            response = requests.get(url, timeout=30)
            response.raise_for_status()
            with open(filepath, 'wb') as f:
                f.write(response.content)
            size_kb = len(response.content) / 1024
            print(f"   ✅ {filename} berhasil diunduh ({size_kb:.1f} KB)")
        except Exception as e:
            print(f"   ❌ Gagal mengunduh {filename}: {e}")
    else:
        print(f"   ✅ {filename} sudah ada, skip download")

# Load dataset utama
print("\n📂 Memuat dataset...")
try:
    df_raw = pd.read_csv("dataset/data.csv", encoding='utf-8')
except UnicodeDecodeError:
    df_raw = pd.read_csv("dataset/data.csv", encoding='latin-1')

print(f"✅ Dataset berhasil dimuat!")
print(f"   Jumlah baris  : {len(df_raw):,}")
print(f"   Jumlah kolom  : {len(df_raw.columns)}")
print(f"   Kolom         : {list(df_raw.columns)}")
print(f"\n📋 Preview 5 baris pertama:")
df_raw.head()

In [ ]:
# ============================================================
# Inspeksi struktur dataset
# ============================================================

print("📊 Informasi Dataset:")
print("=" * 50)
df_raw.info()
print()

# Cek nilai unik di setiap kolom label
print("\n🏷️  Nilai unik per kolom:")
for col in df_raw.columns:
    unique_vals = df_raw[col].unique()
    print(f"  {col}: {unique_vals[:10]}")

print(f"\n🔍 Cek missing values:")
print(df_raw.isnull().sum())

---
## 🔍 LANGKAH 2 — EDA (Exploratory Data Analysis)

Menganalisis:
- Distribusi label asli (multi-label)
- Panjang teks (karakter & kata)
- Frekuensi kata yang sering muncul
- Potensi ketidakseimbangan kelas (class imbalance)

In [ ]:
# ============================================================
# LANGKAH 2A: Analisis Distribusi Label
# ============================================================

# Identifikasi kolom teks dan label
# Dataset okkyibrohim memiliki kolom: Tweet, HS, Abusive, HS_Individual, 
# HS_Group, HS_Religion, HS_Race, HS_Physical, HS_Gender, HS_Other,
# HS_Weak, HS_Moderate, HS_Strong

# Identifikasi kolom teks secara otomatis
text_col = None
for col in df_raw.columns:
    if df_raw[col].dtype == object and df_raw[col].str.len().mean() > 20:
        text_col = col
        break

if text_col is None:
    text_col = df_raw.columns[0]  # fallback ke kolom pertama

print(f"✅ Kolom teks terdeteksi: '{text_col}'")

# Kolom label yang tersedia
label_cols = [col for col in df_raw.columns if col != text_col]
print(f"✅ Kolom label: {label_cols}")

# Visualisasi distribusi label
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Distribusi Label Dataset AI SHIELD', fontsize=16, fontweight='bold')

colors = ['#4CAF50', '#f44336', '#2196F3', '#FF9800', '#9C27B0', '#00BCD4']

for idx, (col, ax, color) in enumerate(zip(label_cols[:6], axes.flatten(), colors)):
    if col in df_raw.columns:
        counts = df_raw[col].value_counts()
        bars = ax.bar(counts.index.astype(str), counts.values, color=[color, '#E0E0E0'][:len(counts)])
        ax.set_title(f'Label: {col}', fontweight='bold')
        ax.set_xlabel('Nilai')
        ax.set_ylabel('Frekuensi')
        for bar, val in zip(bars, counts.values):
            ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 10,
                    f'{val:,}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('eda_label_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Grafik distribusi label disimpan ke 'eda_label_distribution.png'")

In [ ]:
# ============================================================
# LANGKAH 2B: Analisis Panjang Teks
# ============================================================

# Drop baris dengan teks kosong
df_raw = df_raw.dropna(subset=[text_col]).reset_index(drop=True)
df_raw[text_col] = df_raw[text_col].astype(str)

# Hitung panjang teks
df_raw['char_length'] = df_raw[text_col].str.len()
df_raw['word_count'] = df_raw[text_col].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Analisis Panjang Teks', fontsize=14, fontweight='bold')

# Distribusi panjang karakter
axes[0].hist(df_raw['char_length'], bins=50, color='#2196F3', alpha=0.7, edgecolor='black')
axes[0].axvline(df_raw['char_length'].mean(), color='red', linestyle='--', label=f"Mean: {df_raw['char_length'].mean():.0f}")
axes[0].axvline(df_raw['char_length'].median(), color='orange', linestyle='--', label=f"Median: {df_raw['char_length'].median():.0f}")
axes[0].set_title('Distribusi Panjang Karakter')
axes[0].set_xlabel('Jumlah Karakter')
axes[0].set_ylabel('Frekuensi')
axes[0].legend()

# Distribusi jumlah kata
axes[1].hist(df_raw['word_count'], bins=50, color='#4CAF50', alpha=0.7, edgecolor='black')
axes[1].axvline(df_raw['word_count'].mean(), color='red', linestyle='--', label=f"Mean: {df_raw['word_count'].mean():.1f}")
axes[1].axvline(df_raw['word_count'].median(), color='orange', linestyle='--', label=f"Median: {df_raw['word_count'].median():.1f}")
axes[1].set_title('Distribusi Jumlah Kata')
axes[1].set_xlabel('Jumlah Kata')
axes[1].set_ylabel('Frekuensi')
axes[1].legend()

plt.tight_layout()
plt.savefig('eda_text_length.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Statistik Panjang Teks:")
print(f"  Panjang karakter — Min: {df_raw['char_length'].min()}, Max: {df_raw['char_length'].max()}, "
      f"Mean: {df_raw['char_length'].mean():.1f}, Median: {df_raw['char_length'].median():.0f}")
print(f"  Jumlah kata      — Min: {df_raw['word_count'].min()}, Max: {df_raw['word_count'].max()}, "
      f"Mean: {df_raw['word_count'].mean():.1f}, Median: {df_raw['word_count'].median():.0f}")
print(f"\n  💡 Rekomendasi max_length tokenizer: 128 token")
print(f"     (95th percentile kata: {df_raw['word_count'].quantile(0.95):.0f} kata)")

In [ ]:
# ============================================================
# LANGKAH 2C: Ringkasan EDA
# ============================================================

print("📋 RINGKASAN EDA")
print("=" * 60)
print(f"  Total sampel data           : {len(df_raw):,} tweet")
print(f"  Kolom teks                  : {text_col}")
print(f"  Jumlah kolom label          : {len(label_cols)}")
print(f"  Rata-rata panjang teks      : {df_raw['char_length'].mean():.1f} karakter")
print(f"  Rata-rata jumlah kata       : {df_raw['word_count'].mean():.1f} kata")
print()

if 'HS' in df_raw.columns:
    hs_dist = df_raw['HS'].value_counts()
    print(f"  Distribusi Hate Speech (HS):")
    for val, count in hs_dist.items():
        pct = count / len(df_raw) * 100
        print(f"    {val}: {count:,} ({pct:.1f}%)")

if 'Abusive' in df_raw.columns:
    ab_dist = df_raw['Abusive'].value_counts()
    print(f"\n  Distribusi Abusive Language:")
    for val, count in ab_dist.items():
        pct = count / len(df_raw) * 100
        print(f"    {val}: {count:,} ({pct:.1f}%)")

print("\n✅ EDA selesai!")

---
## 🧹 LANGKAH 3 — Preprocessing Teks

Tahap preprocessing sesuai paper Ibrohim & Budi (2019) dan kebutuhan AI SHIELD:
1. **Lowercase** — semua huruf menjadi huruf kecil
2. **Hapus noise** — RT, username (@), URL, hashtag (#), karakter khusus
3. **Normalisasi slang** — menggunakan kamus alay (new_kamusalay.csv)
4. **Hapus spasi berlebih** — normalisasi whitespace

> **Catatan:** Stemming dan stopword removal **tidak dilakukan** karena IndoBERT membutuhkan teks yang lebih natural untuk memahami konteks kalimat secara penuh.

In [ ]:
# ============================================================
# LANGKAH 3A: Load Kamus Alay (Normalisasi Slang)
# ============================================================

# Load kamus normalisasi alay
try:
    alay_df = pd.read_csv("dataset/alay_dict.csv", encoding='utf-8', header=None)
except:
    try:
        alay_df = pd.read_csv("dataset/alay_dict.csv", encoding='latin-1', header=None)
    except:
        print("⚠️  File kamus alay tidak ditemukan. Membuat kamus minimal...")
        alay_data = [
            ['gak', 'tidak'], ['ga', 'tidak'], ['g', 'tidak'], ['tdk', 'tidak'],
            ['yg', 'yang'], ['dgn', 'dengan'], ['utk', 'untuk'], ['krn', 'karena'],
            ['sdh', 'sudah'], ['blm', 'belum'], ['lg', 'lagi'], ['jg', 'juga'],
            ['bgt', 'banget'], ['bk', 'bukan'], ['msh', 'masih'], ['sm', 'sama'],
            ['pd', 'pada'], ['dr', 'dari'], ['sy', 'saya'], ['km', 'kamu'],
            ['aku', 'aku'], ['tp', 'tetapi'], ['klo', 'kalau'], ['kl', 'kalau'],
            ['emg', 'memang'], ['udah', 'sudah'], ['loh', 'loh'], ['dong', 'dong'],
            ['nih', 'ini'], ['tuh', 'itu'], ['dah', 'sudah'], ['enggak', 'tidak'],
        ]
        alay_df = pd.DataFrame(alay_data)

alay_df.columns = ['slang', 'formal'] if len(alay_df.columns) >= 2 else alay_df.columns
alay_df = alay_df.dropna()
alay_dict = dict(zip(alay_df.iloc[:, 0].str.lower(), alay_df.iloc[:, 1].str.lower()))

print(f"✅ Kamus alay dimuat: {len(alay_dict):,} kata")
print(f"   Contoh: {dict(list(alay_dict.items())[:5])}")

In [ ]:
# ============================================================
# LANGKAH 3B: Fungsi Preprocessing
# ============================================================

def normalize_alay(text, alay_dict):
    """Normalisasi kata-kata slang/alay menggunakan kamus."""
    words = text.split()
    normalized = [alay_dict.get(word, word) for word in words]
    return ' '.join(normalized)


def preprocess_text(text, alay_dict=None):
    """
    Pipeline preprocessing teks untuk AI SHIELD.
    
    Langkah:
    1. Lowercase
    2. Hapus RT, username, URL, hashtag
    3. Hapus emoji dan karakter non-ASCII
    4. Hapus karakter khusus & angka berlebih
    5. Normalisasi slang (opsional)
    6. Hapus spasi berlebih
    
    Args:
        text (str): Teks mentah
        alay_dict (dict): Kamus normalisasi slang
    
    Returns:
        str: Teks yang sudah diproses
    """
    if not isinstance(text, str):
        return ""
    
    # 1. Lowercase
    text = text.lower()
    
    # 2. Hapus RT (retweet symbol)
    text = re.sub(r'\brt\b', '', text)
    
    # 3. Hapus username Twitter (@username)
    text = re.sub(r'@[\w]+', '', text)
    
    # 4. Hapus URL (http/https)
    text = re.sub(r'http\S+|www\.\S+|https\S+', '', text)
    
    # 5. Hapus hashtag (#topik)
    text = re.sub(r'#[\w]+', '', text)
    
    # 6. Hapus emoji dan karakter non-ASCII
    text = text.encode('ascii', 'ignore').decode('ascii')
    
    # 7. Hapus tanda baca berlebih (pertahankan tanda baca dasar)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    
    # 8. Hapus angka yang berdiri sendiri (bukan bagian kata)
    text = re.sub(r'\b\d+\b', '', text)
    
    # 9. Normalisasi slang
    if alay_dict:
        text = normalize_alay(text, alay_dict)
    
    # 10. Hapus spasi berlebih
    text = ' '.join(text.split())
    
    return text.strip()


# Test fungsi preprocessing
test_texts = [
    "RT @user123: Anjing banget sih lo!! http://spam.com #politik",
    "Diskusi ini bagus bgt, makasih yg udah share 😊👍",
    "@mahasiswa dasar goblok, ga punya otak",
    "Terima kasih atas penjelasannya yang sangat informatif!",
]

print("🧪 Test Preprocessing:")
print("=" * 60)
for text in test_texts:
    processed = preprocess_text(text, alay_dict)
    print(f"  Asli     : {text}")
    print(f"  Processed: {processed}")
    print()

In [ ]:
# ============================================================
# LANGKAH 3C: Terapkan Preprocessing ke Seluruh Dataset
# ============================================================

print("⚙️  Memproses seluruh dataset... (mungkin memerlukan beberapa menit)")

df_raw['text_clean'] = df_raw[text_col].apply(lambda x: preprocess_text(x, alay_dict))

# Hapus baris dengan teks kosong setelah preprocessing
n_before = len(df_raw)
df_raw = df_raw[df_raw['text_clean'].str.len() > 3].reset_index(drop=True)
n_after = len(df_raw)

print(f"✅ Preprocessing selesai!")
print(f"   Baris sebelum: {n_before:,}")
print(f"   Baris sesudah: {n_after:,}")
print(f"   Dihapus      : {n_before - n_after:,} baris (teks terlalu pendek)")

print(f"\n📋 Contoh hasil preprocessing:")
df_raw[[text_col, 'text_clean']].head(5)

---
## 🏷️ LANGKAH 4 — Relabeling Dataset (Multi-Label → Biner)

Konversi label multi-kelas asli ke **2 kelas biner** sesuai kebutuhan AI SHIELD:

| Label Asli | Label Baru | Keterangan |
|-----------|-----------|------------|
| `HS=0` dan `Abusive=0` | `PANTAS` (0) | Tweet non-hate dan tidak kasar |
| `HS=1` ATAU `Abusive=1` | `TIDAK PANTAS` (1) | Tweet hate speech atau mengandung bahasa kasar |

**Kebijakan Zero-Tolerance:** Setiap tweet yang mengandung hate speech **atau** bahasa kasar (meski hanya salah satu) dikategorikan sebagai `TIDAK PANTAS`.

In [ ]:
# ============================================================
# LANGKAH 4: Relabeling ke Label Biner
# ============================================================

def create_binary_label(row):
    """
    Membuat label biner dari kolom-kolom label dataset.
    
    Logika zero-tolerance:
    - Jika ada kolom HS (Hate Speech) atau Abusive bernilai 1 → TIDAK PANTAS
    - Jika semua kolom tersebut bernilai 0 → PANTAS
    
    Returns:
        int: 1 (TIDAK PANTAS) atau 0 (PANTAS)
    """
    # Kolom yang menandakan konten tidak pantas
    unsafe_cols = ['HS', 'Abusive', 'HS_Individual', 'HS_Group',
                   'HS_Religion', 'HS_Race', 'HS_Physical', 'HS_Gender',
                   'HS_Other', 'HS_Weak', 'HS_Moderate', 'HS_Strong']
    
    for col in unsafe_cols:
        if col in row.index:
            try:
                if int(row[col]) == 1:
                    return 1  # TIDAK PANTAS
            except (ValueError, TypeError):
                pass
    
    return 0  # PANTAS


# Terapkan relabeling
df_raw['label'] = df_raw.apply(create_binary_label, axis=1)
df_raw['label_text'] = df_raw['label'].map({0: 'PANTAS', 1: 'TIDAK PANTAS'})

# Hitung distribusi label baru
label_dist = df_raw['label'].value_counts()
total = len(df_raw)

print("✅ Relabeling selesai!")
print()
print("📊 Distribusi Label Biner:")
print("=" * 40)
for label_val, count in label_dist.items():
    label_name = 'PANTAS' if label_val == 0 else 'TIDAK PANTAS'
    pct = count / total * 100
    bar = '█' * int(pct / 2)
    print(f"  {label_name:15s} ({label_val}): {count:,} ({pct:.1f}%) {bar}")

# Visualisasi
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Distribusi Label Biner (setelah Relabeling)', fontsize=14, fontweight='bold')

# Bar chart
colors_bar = ['#4CAF50', '#f44336']
bars = axes[0].bar(['PANTAS\n(0)', 'TIDAK PANTAS\n(1)'],
                   [label_dist.get(0, 0), label_dist.get(1, 0)],
                   color=colors_bar, edgecolor='black', linewidth=0.5)
axes[0].set_title('Count per Label')
axes[0].set_ylabel('Jumlah Data')
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 30,
                 f'{bar.get_height():,}', ha='center', va='bottom', fontweight='bold')

# Pie chart
pie_labels = [f'PANTAS\n{label_dist.get(0, 0):,}', f'TIDAK PANTAS\n{label_dist.get(1, 0):,}']
axes[1].pie([label_dist.get(0, 0), label_dist.get(1, 0)],
            labels=pie_labels, colors=colors_bar, autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 11})
axes[1].set_title('Proporsi per Label')

plt.tight_layout()
plt.savefig('label_binary_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Cek class imbalance
ratio = label_dist.get(0, 1) / label_dist.get(1, 1)
if ratio > 3 or ratio < 0.33:
    print(f"\n⚠️  Class imbalance terdeteksi! Rasio PANTAS:TIDAK PANTAS = {ratio:.1f}:1")
    print("   Pertimbangkan: class_weight='balanced' atau oversampling kelas minoritas")
else:
    print(f"\n✅ Distribusi kelas cukup seimbang (rasio {ratio:.1f}:1)")

---
## ✂️ LANGKAH 5 — Split Dataset

Pembagian dataset menggunakan **Stratified Split** untuk memastikan proporsi label seimbang di semua split:
- **Train**: 70% — untuk melatih model
- **Validation**: 15% — untuk memantau performa selama training
- **Test**: 15% — untuk evaluasi final yang tidak bias

In [ ]:
# ============================================================
# LANGKAH 5: Stratified Train/Validation/Test Split
# ============================================================

# Siapkan data
df_model = df_raw[['text_clean', 'label', 'label_text']].copy()
df_model = df_model.dropna(subset=['text_clean']).reset_index(drop=True)

X = df_model['text_clean'].values
y = df_model['label'].values

# Split 1: Train (70%) vs Temp (30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)

# Split 2: Validation (15%) vs Test (15%) dari Temp
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print("✅ Dataset berhasil dibagi:")
print()
print(f"  {'Split':<12} {'Total':>8} {'PANTAS':>10} {'TIDAK PANTAS':>15}")
print("  " + "-" * 47)

for split_name, X_split, y_split in [
    ('Train (70%)', X_train, y_train),
    ('Validation (15%)', X_val, y_val),
    ('Test (15%)', X_test, y_test)
]:
    n_total = len(y_split)
    n_pantas = (y_split == 0).sum()
    n_tidak = (y_split == 1).sum()
    pct_pantas = n_pantas / n_total * 100
    pct_tidak = n_tidak / n_total * 100
    print(f"  {split_name:<20} {n_total:>6,}  "
          f"{n_pantas:>5,} ({pct_pantas:.1f}%)  "
          f"{n_tidak:>5,} ({pct_tidak:.1f}%)")

print(f"\n  Total Dataset: {len(df_model):,} sampel")
print("\n✅ Stratified split berhasil — proporsi label terjaga di semua split!")

---
## 🤖 LANGKAH 6 — Fine-tuning IndoBERT

Menggunakan model **`indobenchmark/indobert-base-p1`** dari HuggingFace:
- Pre-trained pada corpus bahasa Indonesia yang besar
- Arsitektur BERT base (12 layer, 768 hidden, 12 heads)
- Fine-tuned untuk **Binary Sequence Classification**

**Hyperparameter yang digunakan:**
- Learning rate: `2e-5`
- Batch size: `16` (train) / `32` (eval)
- Epochs: `5` (dengan early stopping)
- Max token length: `128`
- Warmup steps: `10%` dari total steps

In [ ]:
# ============================================================
# LANGKAH 6A: Setup Tokenizer IndoBERT
# ============================================================

MODEL_NAME = "indobenchmark/indobert-base-p1"
MAX_LENGTH = 128

print(f"⬇️  Memuat tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"✅ Tokenizer berhasil dimuat!")
print(f"   Vocab size    : {tokenizer.vocab_size:,}")
print(f"   Max length    : {MAX_LENGTH} token")

# Test tokenisasi
sample_text = "Diskusi yang sangat menarik dan informatif"
tokens = tokenizer(sample_text, return_tensors='pt')
print(f"\n🧪 Test tokenisasi:")
print(f"   Input  : '{sample_text}'")
print(f"   Tokens : {tokenizer.convert_ids_to_tokens(tokens['input_ids'][0])}")
print(f"   Length : {tokens['input_ids'].shape[1]} token")

In [ ]:
# ============================================================
# LANGKAH 6B: Buat PyTorch Dataset Class
# ============================================================

class AIShieldDataset(Dataset):
    """
    PyTorch Dataset untuk fine-tuning IndoBERT pada task
    klasifikasi biner hate speech & abusive language.
    """
    
    def __init__(self, texts, labels, tokenizer, max_length=128):
        """
        Args:
            texts (list): Daftar teks yang sudah dipreprocess
            labels (list): Daftar label (0=PANTAS, 1=TIDAK PANTAS)
            tokenizer: HuggingFace tokenizer
            max_length (int): Panjang maksimum token
        """
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])
        
        # Tokenisasi
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(label, dtype=torch.long)
        }


# Buat dataset objects
train_dataset = AIShieldDataset(X_train, y_train, tokenizer, MAX_LENGTH)
val_dataset   = AIShieldDataset(X_val,   y_val,   tokenizer, MAX_LENGTH)
test_dataset  = AIShieldDataset(X_test,  y_test,  tokenizer, MAX_LENGTH)

print("✅ Dataset PyTorch berhasil dibuat:")
print(f"   Train     : {len(train_dataset):,} sampel")
print(f"   Validation: {len(val_dataset):,} sampel")
print(f"   Test      : {len(test_dataset):,} sampel")

# Cek satu sampel
sample = train_dataset[0]
print(f"\n📋 Cek satu sampel dari train dataset:")
print(f"   input_ids shape     : {sample['input_ids'].shape}")
print(f"   attention_mask shape: {sample['attention_mask'].shape}")
print(f"   label               : {sample['labels'].item()} ({'PANTAS' if sample['labels'].item()==0 else 'TIDAK PANTAS'})")

In [ ]:
# ============================================================
# LANGKAH 6C: Load Model IndoBERT + Setup Training
# ============================================================

# Label mapping
id2label = {0: "PANTAS", 1: "TIDAK PANTAS"}
label2id = {"PANTAS": 0, "TIDAK PANTAS": 1}

print(f"⬇️  Memuat model IndoBERT: {MODEL_NAME}")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)
model = model.to(device)

# Hitung jumlah parameter
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✅ Model IndoBERT berhasil dimuat!")
print(f"   Total parameter   : {total_params:,}")
print(f"   Parameter trainable: {trainable_params:,}")
print(f"   Model ada di      : {next(model.parameters()).device}")

In [ ]:
# ============================================================
# LANGKAH 6D: Definisi Fungsi Metrik Evaluasi
# ============================================================

def compute_metrics(eval_pred):
    """
    Fungsi metrik yang dipanggil oleh HuggingFace Trainer
    setiap akhir epoch evaluasi.
    
    Menghitung: Accuracy, Precision, Recall, F1 (macro & per-kelas)
    """
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    # Hitung metrik per-kelas
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average=None, labels=[0, 1]
    )
    
    # Hitung metrik macro
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, predictions, average='macro'
    )
    
    acc = accuracy_score(labels, predictions)
    
    return {
        'accuracy': acc,
        'f1_macro': f1_macro,
        'f1_pantas': f1[0],
        'f1_tidak_pantas': f1[1],
        'precision_tidak_pantas': precision[1],
        'recall_tidak_pantas': recall[1],
        'precision_macro': p_macro,
        'recall_macro': r_macro,
    }

print("✅ Fungsi metrik evaluasi siap!")
print("   Metrik yang dihitung: accuracy, f1_macro, f1_pantas, f1_tidak_pantas")
print("   Metrik target:")
print("     - Accuracy ≥ 85%")
print("     - F1 TIDAK PANTAS ≥ 82%")

In [ ]:
# ============================================================
# LANGKAH 6E: Setup Training Arguments & Trainer
# ============================================================

# Hitung total training steps untuk warmup
BATCH_SIZE_TRAIN = 16
NUM_EPOCHS = 5
steps_per_epoch = len(train_dataset) // BATCH_SIZE_TRAIN
total_steps = steps_per_epoch * NUM_EPOCHS
warmup_steps = int(total_steps * 0.1)  # 10% dari total steps

print(f"📊 Info Training:")
print(f"   Batch size (train)   : {BATCH_SIZE_TRAIN}")
print(f"   Jumlah epoch         : {NUM_EPOCHS}")
print(f"   Steps per epoch      : {steps_per_epoch:,}")
print(f"   Total training steps : {total_steps:,}")
print(f"   Warmup steps         : {warmup_steps:,}")

training_args = TrainingArguments(
    # === Direktori output ===
    output_dir="./ai_shield_checkpoints",
    overwrite_output_dir=True,
    
    # === Hyperparameter ===
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE_TRAIN,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=warmup_steps,
    
    # === Evaluasi & Logging ===
    evaluation_strategy="epoch",  # Evaluasi setiap akhir epoch
    save_strategy="epoch",         # Simpan checkpoint setiap epoch
    logging_strategy="steps",
    logging_steps=50,
    load_best_model_at_end=True,   # Load model terbaik di akhir
    metric_for_best_model="f1_tidak_pantas",  # Optimasi berdasarkan F1 TIDAK PANTAS
    greater_is_better=True,
    
    # === Pengaturan Teknis ===
    fp16=torch.cuda.is_available(),  # Mixed precision training (hanya di GPU)
    dataloader_num_workers=2,
    seed=SEED,
    report_to="none",  # Matikan wandb/tensorboard
)

# Setup Trainer dengan Early Stopping
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]  # Stop jika tidak membaik 2 epoch
)

print(f"\n✅ Trainer siap! Memulai training...")

In [ ]:
# ============================================================
# LANGKAH 6F: Jalankan Fine-tuning
# ============================================================

print("🚀 Memulai Fine-tuning IndoBERT...")
print("   Estimasi waktu: 20-60 menit (tergantung GPU)")
print("   Early stopping aktif (patience=2 epoch)")
print()

# Mulai training
train_result = trainer.train()

print()
print("✅ Training selesai!")
print(f"   Total waktu training : {train_result.metrics['train_runtime']:.0f} detik")
print(f"   Training loss akhir  : {train_result.metrics['train_loss']:.4f}")
print(f"   Samples per second   : {train_result.metrics['train_samples_per_second']:.1f}")

In [ ]:
# ============================================================
# LANGKAH 6G: Visualisasi Riwayat Training
# ============================================================

# Ambil riwayat training
history = trainer.state.log_history
train_logs = [x for x in history if 'loss' in x and 'eval_loss' not in x]
eval_logs  = [x for x in history if 'eval_loss' in x]

if eval_logs:
    epochs_eval = [x['epoch'] for x in eval_logs]
    eval_loss   = [x.get('eval_loss', None) for x in eval_logs]
    eval_acc    = [x.get('eval_accuracy', None) for x in eval_logs]
    eval_f1_tp  = [x.get('eval_f1_tidak_pantas', None) for x in eval_logs]
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Riwayat Training IndoBERT — AI SHIELD', fontsize=14, fontweight='bold')
    
    # Plot Loss
    axes[0].plot(epochs_eval, eval_loss, 'b-o', label='Validation Loss')
    axes[0].set_title('Validation Loss per Epoch')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Plot Accuracy
    if any(v is not None for v in eval_acc):
        axes[1].plot(epochs_eval, eval_acc, 'g-o', label='Accuracy')
        axes[1].axhline(y=0.85, color='r', linestyle='--', alpha=0.7, label='Target 85%')
        axes[1].set_title('Validation Accuracy per Epoch')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Accuracy')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
        axes[1].set_ylim(0, 1)
    
    # Plot F1 TIDAK PANTAS
    if any(v is not None for v in eval_f1_tp):
        axes[2].plot(epochs_eval, eval_f1_tp, 'r-o', label='F1 TIDAK PANTAS')
        axes[2].axhline(y=0.82, color='orange', linestyle='--', alpha=0.7, label='Target 82%')
        axes[2].set_title('F1-Score TIDAK PANTAS per Epoch')
        axes[2].set_xlabel('Epoch')
        axes[2].set_ylabel('F1-Score')
        axes[2].legend()
        axes[2].grid(True, alpha=0.3)
        axes[2].set_ylim(0, 1)
    
    plt.tight_layout()
    plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Grafik riwayat training disimpan ke 'training_history.png'")
else:
    print("⚠️  Tidak ada riwayat evaluasi untuk divisualisasikan")

---
## 📊 LANGKAH 7 — Evaluasi Model pada Test Set

Evaluasi final menggunakan **test set yang belum pernah dilihat model** selama training:
- **Accuracy** — proporsi prediksi benar dari seluruh data
- **Precision** — dari semua yang diprediksi TIDAK PANTAS, berapa yang benar
- **Recall** — dari semua yang benar-benar TIDAK PANTAS, berapa yang terdeteksi
- **F1-Score** — harmonic mean dari Precision dan Recall
- **Confusion Matrix** — visualisasi lengkap prediksi vs aktual

In [ ]:
# ============================================================
# LANGKAH 7A: Evaluasi pada Test Set
# ============================================================

print("📊 Mengevaluasi model pada test set...")

# Dapatkan prediksi
test_predictions = trainer.predict(test_dataset)
y_pred_logits = test_predictions.predictions
y_pred = np.argmax(y_pred_logits, axis=-1)
y_true = test_predictions.label_ids

# Hitung metrik
acc = accuracy_score(y_true, y_pred)
p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(y_true, y_pred, average='macro')
p_per_class, r_per_class, f1_per_class, support = precision_recall_fscore_support(y_true, y_pred, average=None, labels=[0, 1])

print()
print("=" * 60)
print("  📋 LAPORAN EVALUASI MODEL — AI SHIELD IndoBERT")
print("=" * 60)
print(f"  Accuracy            : {acc:.4f} ({acc*100:.2f}%) {'✅' if acc >= 0.85 else '❌ (target ≥ 85%)'}") 
print(f"  F1-Score (Macro)    : {f1_macro:.4f} ({f1_macro*100:.2f}%)")
print(f"  Precision (Macro)   : {p_macro:.4f}")
print(f"  Recall (Macro)      : {r_macro:.4f}")
print()
print(f"  === Per Kelas ===")
print(f"  {'Kelas':<20} {'Precision':>10} {'Recall':>8} {'F1-Score':>10} {'Support':>10}")
print("  " + "-" * 62)
for i, label_name in enumerate(['PANTAS', 'TIDAK PANTAS']):
    f1_check = '✅' if (i == 1 and f1_per_class[i] >= 0.82) else ('❌' if i == 1 else '')
    print(f"  {label_name:<20} {p_per_class[i]:>10.4f} {r_per_class[i]:>8.4f} "
          f"{f1_per_class[i]:>10.4f} {support[i]:>10,} {f1_check}")
print("=" * 60)

# Evaluasi target
print()
print("🎯 Evaluasi Target Performa:")
target_acc = acc >= 0.85
target_f1  = f1_per_class[1] >= 0.82
print(f"  ✅ Accuracy ≥ 85%              : {'TERCAPAI ✅' if target_acc else 'BELUM TERCAPAI ❌'}  ({acc*100:.2f}%)")
print(f"  ✅ F1 TIDAK PANTAS ≥ 82%       : {'TERCAPAI ✅' if target_f1 else 'BELUM TERCAPAI ❌'}  ({f1_per_class[1]*100:.2f}%)")

if not target_acc or not target_f1:
    print()
    print("💡 Saran jika target belum tercapai:")
    print("   - Tambah epoch training (ubah NUM_EPOCHS menjadi 7-10)")
    print("   - Lakukan augmentasi data kelas TIDAK PANTAS")
    print("   - Coba learning rate lebih kecil (1e-5)")
    print("   - Gunakan model yang lebih besar (indobert-large)")

In [ ]:
# ============================================================
# LANGKAH 7B: Visualisasi Confusion Matrix
# ============================================================

cm = confusion_matrix(y_true, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Confusion Matrix — AI SHIELD IndoBERT', fontsize=14, fontweight='bold')

class_names = ['PANTAS', 'TIDAK\nPANTAS']

# Confusion Matrix (count)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            ax=axes[0], linewidths=0.5, linecolor='gray')
axes[0].set_title('Confusion Matrix (Jumlah)', fontweight='bold')
axes[0].set_xlabel('Prediksi', fontweight='bold')
axes[0].set_ylabel('Aktual', fontweight='bold')

# Confusion Matrix (persentase)
cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names,
            ax=axes[1], linewidths=0.5, linecolor='gray')
axes[1].set_title('Confusion Matrix (Persentase %)', fontweight='bold')
axes[1].set_xlabel('Prediksi', fontweight='bold')
axes[1].set_ylabel('Aktual', fontweight='bold')

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Interpretasi
tn, fp, fn, tp = cm.ravel()
print(f"\n📋 Interpretasi Confusion Matrix:")
print(f"   True Positive  (TP): {tp:,} — TIDAK PANTAS yang berhasil dideteksi")
print(f"   True Negative  (TN): {tn:,} — PANTAS yang benar tidak diblokir")
print(f"   False Positive (FP): {fp:,} — PANTAS yang salah diblokir (false alarm)")
print(f"   False Negative (FN): {fn:,} — TIDAK PANTAS yang lolos (tidak terdeteksi)")
print(f"\n   ⚠️  Kasus paling berbahaya: False Negative (FN) — konten tidak pantas yang lolos")

In [ ]:
# ============================================================
# LANGKAH 7C: Classification Report Lengkap
# ============================================================

print("📋 Classification Report Lengkap:")
print("=" * 60)
print(classification_report(y_true, y_pred, target_names=['PANTAS', 'TIDAK PANTAS']))

---
## 🎯 LANGKAH 8 — Kalibrasi Confidence Threshold

Menentukan **threshold optimal** untuk mengklasifikasikan pesan sebagai `TIDAK PANTAS`.

Strategi AI SHIELD:
- **Default threshold: 0.75** — cukup tinggi untuk menghindari false positive berlebihan
- Confidence < 0.75 → tetap disembunyikan tapi pengirim dapat ajukan review ke admin
- Analisis kurva F1 vs threshold untuk memvalidasi pilihan threshold

In [ ]:
# ============================================================
# LANGKAH 8: Kalibrasi Confidence Threshold
# ============================================================

import torch.nn.functional as F

# Hitung probabilitas menggunakan softmax
logits_tensor = torch.tensor(y_pred_logits)
probs = F.softmax(logits_tensor, dim=-1).numpy()
prob_tidak_pantas = probs[:, 1]  # Probabilitas kelas TIDAK PANTAS

# Analisis threshold
thresholds = np.arange(0.3, 0.95, 0.05)
results_threshold = []

for thresh in thresholds:
    y_pred_thresh = (prob_tidak_pantas >= thresh).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred_thresh, average=None, labels=[0, 1], zero_division=0
    )
    acc_thresh = accuracy_score(y_true, y_pred_thresh)
    results_threshold.append({
        'threshold': thresh,
        'accuracy': acc_thresh,
        'f1_tidak_pantas': f1[1],
        'precision_tidak_pantas': p[1],
        'recall_tidak_pantas': r[1],
        'f1_macro': (f1[0] + f1[1]) / 2
    })

df_thresh = pd.DataFrame(results_threshold)

# Visualisasi
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(df_thresh['threshold'], df_thresh['accuracy'], 'b-o', label='Accuracy', linewidth=2)
ax.plot(df_thresh['threshold'], df_thresh['f1_tidak_pantas'], 'r-s', label='F1 TIDAK PANTAS', linewidth=2)
ax.plot(df_thresh['threshold'], df_thresh['precision_tidak_pantas'], 'g-^', label='Precision TIDAK PANTAS', linewidth=2)
ax.plot(df_thresh['threshold'], df_thresh['recall_tidak_pantas'], 'm-v', label='Recall TIDAK PANTAS', linewidth=2)

ax.axvline(x=0.75, color='orange', linestyle='--', linewidth=2, label='Default Threshold (0.75)')
ax.axhline(y=0.85, color='blue', linestyle=':', alpha=0.5, label='Target Accuracy 85%')
ax.axhline(y=0.82, color='red', linestyle=':', alpha=0.5, label='Target F1 82%')

ax.set_title('Metrik vs Confidence Threshold — AI SHIELD', fontsize=14, fontweight='bold')
ax.set_xlabel('Threshold', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.legend(loc='lower left', fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(0.25, 1.0)
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('threshold_calibration.png', dpi=150, bbox_inches='tight')
plt.show()

# Tampilkan tabel hasil
print("\n📊 Metrik per Threshold:")
print(df_thresh.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

# Threshold pada default 0.75
idx_075 = (df_thresh['threshold'] - 0.75).abs().idxmin()
row_075 = df_thresh.iloc[idx_075]
print(f"\n🎯 Performa pada threshold default (0.75):")
print(f"   Accuracy         : {row_075['accuracy']:.4f} ({row_075['accuracy']*100:.2f}%)")
print(f"   F1 TIDAK PANTAS  : {row_075['f1_tidak_pantas']:.4f} ({row_075['f1_tidak_pantas']*100:.2f}%)")
print(f"   Precision        : {row_075['precision_tidak_pantas']:.4f}")
print(f"   Recall           : {row_075['recall_tidak_pantas']:.4f}")

# Rekomendasi threshold optimal
optimal_idx = df_thresh['f1_tidak_pantas'].idxmax()
optimal_thresh = df_thresh.iloc[optimal_idx]['threshold']
print(f"\n💡 Threshold optimal (F1 tertinggi): {optimal_thresh:.2f}")
print(f"   AI SHIELD menggunakan: 0.75 (default konservatif untuk akademik)")

# Simpan threshold yang digunakan
CONFIDENCE_THRESHOLD = 0.75
print(f"\n✅ Confidence threshold ditetapkan: {CONFIDENCE_THRESHOLD}")

---
## 🔧 LANGKAH 9 — Inference Function

Membuat fungsi `predict(text)` yang akan dipanggil oleh Backend Engineer.  
Interface yang disepakati:

```python
result = predict("teks pesan dari pengguna")
# Returns: {"label": "PANTAS" | "TIDAK PANTAS", "confidence": 0.0–1.0}
```

In [ ]:
# ============================================================
# LANGKAH 9: Fungsi Inferensi Production-Ready
# ============================================================

import torch.nn.functional as F

# Pastikan model dalam mode evaluasi
model.eval()

def predict(text: str, threshold: float = 0.75) -> dict:
    """
    Fungsi inferensi AI SHIELD.
    Mengklasifikasikan teks ke dalam PANTAS atau TIDAK PANTAS.
    
    Interface ini identik dengan classifier.py placeholder di backend,
    sehingga penggantian seamless tanpa mengubah kode backend.
    
    Args:
        text (str): Teks pesan dari pengguna (belum dipreprocess)
        threshold (float): Confidence threshold untuk klasifikasi TIDAK PANTAS
                          Default: 0.75 (sesuai kalibrasi AI SHIELD)
    
    Returns:
        dict: {
            "label": str — "PANTAS" atau "TIDAK PANTAS",
            "confidence": float — nilai confidence (0.0 – 1.0),
            "prob_pantas": float — probabilitas kelas PANTAS,
            "prob_tidak_pantas": float — probabilitas kelas TIDAK PANTAS
        }
    """
    # 1. Preprocessing
    text_clean = preprocess_text(text, alay_dict)
    
    # Handle teks kosong setelah preprocessing
    if not text_clean:
        return {
            "label": "PANTAS",
            "confidence": 0.5,
            "prob_pantas": 0.5,
            "prob_tidak_pantas": 0.5
        }
    
    # 2. Tokenisasi
    inputs = tokenizer(
        text_clean,
        return_tensors='pt',
        max_length=MAX_LENGTH,
        padding='max_length',
        truncation=True
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # 3. Inferensi
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = F.softmax(logits, dim=-1).squeeze().cpu().numpy()
    
    prob_pantas = float(probs[0])
    prob_tidak_pantas = float(probs[1])
    
    # 4. Klasifikasi berdasarkan threshold
    if prob_tidak_pantas >= threshold:
        label = "TIDAK PANTAS"
        confidence = prob_tidak_pantas
    else:
        label = "PANTAS"
        confidence = prob_pantas
    
    return {
        "label": label,
        "confidence": round(confidence, 4),
        "prob_pantas": round(prob_pantas, 4),
        "prob_tidak_pantas": round(prob_tidak_pantas, 4)
    }


# ============================================================
# Uji Fungsi Inferensi
# ============================================================

test_messages = [
    # Pesan PANTAS
    "Halo teman-teman, selamat pagi! Ada yang mau diskusi tentang materi ujian?",
    "Terima kasih atas penjelasannya yang sangat membantu dan informatif.",
    "Semangat belajarnya! Kita bisa kalau usaha bersama.",
    "Apakah ada yang bisa bantu jelaskan konsep rekursif di Python?",
    
    # Pesan TIDAK PANTAS
    "Anjing banget sih, ga ngerti-ngerti juga lo!",
    "Dasar bodoh, mending ga usah masuk kuliah!",
    "Goblok lo semua, otaknya isi apa coba?",
    "Mending belajar dulu sebelum komentar, tolol!",
    
    # Ambiguous
    "Wah anjir, presentasinya keren banget!",  # Slang positif tapi pakai kata kasar
    "Serius dah, ini susah banget materinya...",
]

print("🧪 Pengujian Inference Function:")
print("=" * 80)
print(f"{'Pesan':<45} {'Label':<15} {'Confidence':>10} {'Prob TP':>8}")
print("-" * 80)

for msg in test_messages:
    result = predict(msg)
    status = '🟢' if result['label'] == 'PANTAS' else '🔴'
    print(f"{status} {msg[:43]:<43} {result['label']:<15} {result['confidence']:>10.4f} {result['prob_tidak_pantas']:>8.4f}")

print("=" * 80)
print(f"\n✅ Fungsi predict() siap digunakan oleh Backend Engineer!")
print(f"   Threshold yang digunakan: {CONFIDENCE_THRESHOLD}")

In [ ]:
# ============================================================
# Fungsi Batch Prediction (opsional, untuk efisiensi)
# ============================================================

def predict_batch(texts: list, threshold: float = 0.75, batch_size: int = 32) -> list:
    """
    Fungsi inferensi batch untuk memproses banyak teks sekaligus.
    Lebih efisien dibanding memanggil predict() satu per satu.
    
    Args:
        texts (list): Daftar teks
        threshold (float): Confidence threshold
        batch_size (int): Ukuran batch
    
    Returns:
        list: Daftar dict hasil prediksi
    """
    model.eval()
    results = []
    
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        batch_clean = [preprocess_text(t, alay_dict) for t in batch_texts]
        
        inputs = tokenizer(
            batch_clean,
            return_tensors='pt',
            max_length=MAX_LENGTH,
            padding='max_length',
            truncation=True
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model(**inputs)
            probs = F.softmax(outputs.logits, dim=-1).cpu().numpy()
        
        for j, (prob, text) in enumerate(zip(probs, batch_texts)):
            prob_tidak = float(prob[1])
            prob_pantas = float(prob[0])
            label = "TIDAK PANTAS" if prob_tidak >= threshold else "PANTAS"
            confidence = prob_tidak if label == "TIDAK PANTAS" else prob_pantas
            results.append({
                "text": text,
                "label": label,
                "confidence": round(confidence, 4),
                "prob_pantas": round(prob_pantas, 4),
                "prob_tidak_pantas": round(prob_tidak, 4)
            })
    
    return results

print("✅ Fungsi predict_batch() siap!")

---
## 💾 LANGKAH 10 — Simpan Model & Export ke Google Drive

Menyimpan:
1. Model weights (pytorch_model.bin) 
2. Konfigurasi model (config.json)
3. Tokenizer
4. Metadata model (threshold, metrik evaluasi)

Kemudian upload ke Google Drive untuk digunakan oleh Backend Engineer.

In [ ]:
# ============================================================
# LANGKAH 10A: Simpan Model Lokal
# ============================================================

MODEL_SAVE_PATH = "./ai_shield_indobert_model"
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)

print(f"💾 Menyimpan model ke: {MODEL_SAVE_PATH}")

# Simpan model dan tokenizer
trainer.save_model(MODEL_SAVE_PATH)
tokenizer.save_pretrained(MODEL_SAVE_PATH)

# Simpan metadata model
metadata = {
    "model_name": "AI SHIELD IndoBERT",
    "base_model": MODEL_NAME,
    "task": "binary-text-classification",
    "labels": {"0": "PANTAS", "1": "TIDAK PANTAS"},
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "max_length": MAX_LENGTH,
    "evaluation_metrics": {
        "accuracy": round(float(acc), 4),
        "f1_macro": round(float(f1_macro), 4),
        "f1_pantas": round(float(f1_per_class[0]), 4),
        "f1_tidak_pantas": round(float(f1_per_class[1]), 4),
        "precision_tidak_pantas": round(float(p_per_class[1]), 4),
        "recall_tidak_pantas": round(float(r_per_class[1]), 4),
    },
    "training_config": {
        "learning_rate": 2e-5,
        "batch_size": BATCH_SIZE_TRAIN,
        "num_epochs": NUM_EPOCHS,
        "max_length": MAX_LENGTH,
        "seed": SEED,
        "early_stopping_patience": 2,
        "metric_for_best_model": "f1_tidak_pantas"
    },
    "dataset": {
        "source": "https://github.com/okkyibrohim/id-multi-label-hate-speech-and-abusive-language-detection",
        "total_samples": len(df_model),
        "train_samples": len(X_train),
        "val_samples": len(X_val),
        "test_samples": len(X_test),
        "split_ratio": "70/15/15"
    },
    "project": "AI SHIELD — PIJAK x IBM SkillsBuild",
    "inference_usage": {
        "example": "from transformers import AutoTokenizer, AutoModelForSequenceClassification",
        "model_id": MODEL_SAVE_PATH
    }
}

with open(f"{MODEL_SAVE_PATH}/model_metadata.json", 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

# Cek file yang tersimpan
saved_files = os.listdir(MODEL_SAVE_PATH)
total_size = sum(os.path.getsize(f"{MODEL_SAVE_PATH}/{f}") for f in saved_files) / (1024**2)

print(f"\n✅ Model berhasil disimpan!")
print(f"   Lokasi    : {MODEL_SAVE_PATH}")
print(f"   File      : {saved_files}")
print(f"   Total size: {total_size:.1f} MB")

In [ ]:
# ============================================================
# LANGKAH 10B: Upload Model ke Google Drive
# ============================================================

from google.colab import drive
import shutil

# Mount Google Drive
drive.mount('/content/drive')

# Buat direktori tujuan di Google Drive
GDRIVE_PATH = "/content/drive/MyDrive/AI_SHIELD/model"
os.makedirs(GDRIVE_PATH, exist_ok=True)

print(f"📤 Upload model ke Google Drive: {GDRIVE_PATH}")

# Copy semua file model ke Google Drive
for filename in os.listdir(MODEL_SAVE_PATH):
    src = f"{MODEL_SAVE_PATH}/{filename}"
    dst = f"{GDRIVE_PATH}/{filename}"
    shutil.copy2(src, dst)
    size_mb = os.path.getsize(dst) / (1024**2)
    print(f"   ✅ {filename} ({size_mb:.1f} MB)")

# Copy gambar visualisasi
GDRIVE_ASSETS = "/content/drive/MyDrive/AI_SHIELD/assets"
os.makedirs(GDRIVE_ASSETS, exist_ok=True)

for img_file in ['eda_label_distribution.png', 'eda_text_length.png',
                  'label_binary_distribution.png', 'training_history.png',
                  'confusion_matrix.png', 'threshold_calibration.png']:
    if os.path.exists(img_file):
        shutil.copy2(img_file, f"{GDRIVE_ASSETS}/{img_file}")
        print(f"   📊 {img_file} disimpan ke assets")

print(f"\n✅ Upload selesai!")
print(f"   Google Drive Path: {GDRIVE_PATH}")
print(f"   Salin path ini ke README.md untuk Backend Engineer")

In [ ]:
# ============================================================
# LANGKAH 10C: Verifikasi — Load Model dari Disk & Test
# ============================================================

print("🔍 Verifikasi model yang tersimpan...")

# Load ulang model dari disk
tokenizer_loaded = AutoTokenizer.from_pretrained(MODEL_SAVE_PATH)
model_loaded = AutoModelForSequenceClassification.from_pretrained(MODEL_SAVE_PATH)
model_loaded = model_loaded.to(device)
model_loaded.eval()

print(f"✅ Model berhasil dimuat dari disk!")

# Test inferensi dengan model yang di-load
def predict_from_loaded(text, model_obj, tok_obj, threshold=0.75):
    text_clean = preprocess_text(text, alay_dict)
    inputs = tok_obj(
        text_clean, return_tensors='pt',
        max_length=MAX_LENGTH, padding='max_length', truncation=True
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model_obj(**inputs)
        probs = F.softmax(outputs.logits, dim=-1).squeeze().cpu().numpy()
    label = "TIDAK PANTAS" if probs[1] >= threshold else "PANTAS"
    return {"label": label, "confidence": round(float(max(probs)), 4)}

verify_texts = [
    "Selamat pagi, bagaimana kabar hari ini?",
    "Dasar bodoh, ga ada gunanya!",
]

print("\n🧪 Verifikasi inferensi dari model yang disimpan:")
for text in verify_texts:
    result = predict_from_loaded(text, model_loaded, tokenizer_loaded)
    status = '🟢' if result['label'] == 'PANTAS' else '🔴'
    print(f"  {status} '{text[:50]}' → {result['label']} ({result['confidence']:.4f})")

print("\n✅ Model tersimpan dan berfungsi dengan benar!")

---
## 🏁 RINGKASAN AKHIR

Notebook ini telah menyelesaikan seluruh pipeline AI/ML untuk **AI SHIELD**:

| No | Langkah | Status |
|----|---------|--------|
| 1 | Setup Environment | ✅ |
| 2 | Download Dataset (okkyibrohim) | ✅ |
| 3 | EDA (Exploratory Data Analysis) | ✅ |
| 4 | Preprocessing Teks | ✅ |
| 5 | Relabeling → Biner (PANTAS / TIDAK PANTAS) | ✅ |
| 6 | Split Dataset (70/15/15 Stratified) | ✅ |
| 7 | Fine-tuning IndoBERT | ✅ |
| 8 | Evaluasi (Accuracy, F1, Confusion Matrix) | ✅ |
| 9 | Kalibrasi Confidence Threshold (0.75) | ✅ |
| 10 | Inference Function `predict(text)` | ✅ |
| 11 | Simpan Model + Upload Google Drive | ✅ |

### 📁 File yang Dihasilkan
```
ai_shield_indobert_model/
├── config.json              ← Konfigurasi model
├── pytorch_model.bin        ← Weights model
├── tokenizer_config.json    ← Konfigurasi tokenizer  
├── vocab.txt                ← Vocabulary tokenizer
├── special_tokens_map.json  ← Special tokens
└── model_metadata.json      ← Metadata & metrik evaluasi

assets/
├── eda_label_distribution.png
├── eda_text_length.png
├── label_binary_distribution.png
├── training_history.png
├── confusion_matrix.png
└── threshold_calibration.png
```

### 🔗 Untuk Backend Engineer
```python
# Cara load model dari Google Drive:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("/path/ke/model")
tokenizer = AutoTokenizer.from_pretrained("/path/ke/model")

# Atau gunakan fungsi predict() yang sudah siap:
result = predict("pesan dari pengguna")
# {"label": "PANTAS"|"TIDAK PANTAS", "confidence": 0.0–1.0}
```

---
*AI SHIELD — PIJAK in collaboration with IBM SkillsBuild × Dicoding*

In [ ]:
# ============================================================
# Ringkasan Akhir — Cetak semua metrik utama
# ============================================================

print("=" * 65)
print("  🛡️  AI SHIELD — HASIL AKHIR PENGEMBANGAN MODEL")
print("=" * 65)
print()
print(f"  Model Base      : {MODEL_NAME}")
print(f"  Task            : Binary Text Classification")
print(f"  Dataset         : okkyibrohim/id-multi-label-hate-speech")
print(f"  Total Sampel    : {len(df_model):,} tweet")
print(f"  Split           : 70% train / 15% val / 15% test")
print()
print("  📊 METRIK EVALUASI (Test Set):")
print(f"  Accuracy          : {acc:.4f} ({acc*100:.2f}%) {'✅ TERCAPAI' if acc >= 0.85 else '❌ BELUM TERCAPAI'}")
print(f"  F1-Score Macro    : {f1_macro:.4f} ({f1_macro*100:.2f}%)")
print(f"  F1 PANTAS         : {f1_per_class[0]:.4f} ({f1_per_class[0]*100:.2f}%)")
print(f"  F1 TIDAK PANTAS   : {f1_per_class[1]:.4f} ({f1_per_class[1]*100:.2f}%) {'✅ TERCAPAI' if f1_per_class[1] >= 0.82 else '❌ BELUM TERCAPAI'}")
print(f"  Precision (TP)    : {p_per_class[1]:.4f}")
print(f"  Recall (TP)       : {r_per_class[1]:.4f}")
print()
print(f"  🎯 Confidence Threshold : {CONFIDENCE_THRESHOLD}")
print(f"  📁 Model Path           : {MODEL_SAVE_PATH}")
print()
print("  🏆 TARGET PROYEK:")
print(f"  Accuracy ≥ 85%         : {'✅ TERCAPAI' if acc >= 0.85 else '❌ BELUM TERCAPAI'}")
print(f"  F1 TIDAK PANTAS ≥ 82%  : {'✅ TERCAPAI' if f1_per_class[1] >= 0.82 else '❌ BELUM TERCAPAI'}")
print()
print("=" * 65)
print("  Model siap digunakan oleh Backend Engineer! 🚀")
print("  Salin link Google Drive ke README.md")
print("=" * 65)